# Differential Equations — Session 35
## Section 7.6: Systems of Linear Differential Equations

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Students should be able to transform a constant-coefficient system into simultaneous algebraic equations; solve for transformed variables; interpret determinant zeros as modal frequencies; formulate coupled-spring equations; compare time plots and phase trajectories; and connect transform methods with matrix-system solutions.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |
|---:|---|
| 0–18 min | Transforming linear systems |
| 18–48 min | Coupled springs |
| 48–68 min | Normal modes and beating |
| 68–82 min | Electrical network |
| 82–90 min | Linearized double pendulum extension |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import quad, solve_ivp
from scipy.signal import fftconvolve
from scipy.linalg import expm, eig
from IPython.display import display, Markdown

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=6, suppress=True)

def unit_step(t, a=0.0):
    t = np.asarray(t)
    return (t >= a).astype(float)

print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Formal theory reference

### Principle 7.6-A — Transforming a system

For

$$
\mathbf x'=A\mathbf x+\mathbf f(t),
\qquad
\mathbf x(0)=\mathbf x_0,
$$

the Laplace transform gives

$$
(sI-A)\mathbf X(s)
=
\mathbf x_0+\mathbf F(s).
$$

Hence

$$
\mathbf X(s)
=
(sI-A)^{-1}
\big[\mathbf x_0+\mathbf F(s)\big].
$$

### Proposition 7.6-B — Poles and system modes

Zeros of

$$
\det(sI-A)
$$

are poles of the transformed solution and correspond to natural modes of the system.

### Model 7.6-C — Coupled springs

For masses $m_1,m_2$ and spring constants $k_1,k_2$,

$$
m_1x_1''
=
-k_1x_1+k_2(x_2-x_1),
$$

$$
m_2x_2''
=
-k_2(x_2-x_1).
$$

### Principle 7.6-D — Linearization

A nonlinear mechanical system may be approximated near equilibrium by replacing nonlinear terms with their first-order Taylor approximations, such as $\sin\theta\approx\theta$.

### Classroom Checkpoint — Transform a Matrix System

For

$$
\mathbf X'=A\mathbf X+\mathbf F(t),
\qquad
\mathbf X(0)=\mathbf X_0,
$$

what algebraic equation is obtained after taking Laplace transforms?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Coupled-spring model

In [ ]:
# Original schematic
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.axis("off")
ax.plot([0.05, 0.15], [0.5, 0.5], linewidth=5)
ax.text(0.08, 0.62, "wall", ha="center")
ax.plot([0.15, 0.35], [0.5, 0.5], linewidth=2)
ax.add_patch(plt.Rectangle((0.35, 0.38), 0.12, 0.24, fill=False, linewidth=2))
ax.plot([0.47, 0.68], [0.5, 0.5], linewidth=2)
ax.add_patch(plt.Rectangle((0.68, 0.38), 0.12, 0.24, fill=False, linewidth=2))
ax.text(0.41, 0.5, r"$m_1$", ha="center", va="center")
ax.text(0.74, 0.5, r"$m_2$", ha="center", va="center")
ax.text(0.25, 0.62, r"$k_1$", ha="center")
ax.text(0.57, 0.62, r"$k_2$", ha="center")
plt.show()

In matrix form,

$$
M\mathbf x''+K\mathbf x=0,
$$

where

$$
M=
\begin{pmatrix}
m_1&0\\
0&m_2
\end{pmatrix},
\qquad
K=
\begin{pmatrix}
k_1+k_2&-k_2\\
-k_2&k_2
\end{pmatrix}.
$$

In [ ]:
def coupled_springs(m1=1.0, m2=1.0, k1=4.0, k2=2.0,
                    x10=1.0, x20=0.0, final_time=30):
    M = np.diag([m1, m2])
    K = np.array([[k1+k2, -k2], [-k2, k2]])
    A = np.block([
        [np.zeros((2,2)), np.eye(2)],
        [-np.linalg.solve(M, K), np.zeros((2,2))]
    ])

    def rhs(t, z):
        return A @ z

    t = np.linspace(0, final_time, 1500)
    sol = solve_ivp(rhs, (0, final_time), [x10, x20, 0, 0],
                    t_eval=t, rtol=1e-9, atol=1e-11)

    eigenvalues = np.linalg.eigvals(np.linalg.solve(M, K))
    frequencies = np.sqrt(np.sort(eigenvalues.real))

    plt.plot(t, sol.y[0], label=r"$x_1$")
    plt.plot(t, sol.y[1], label=r"$x_2$")
    plt.xlabel("t")
    plt.ylabel("displacement")
    plt.title("Coupled-spring motion")
    plt.legend()
    plt.show()

    plt.plot(sol.y[0], sol.y[1])
    plt.xlabel(r"$x_1$")
    plt.ylabel(r"$x_2$")
    plt.title("Configuration-space trajectory")
    plt.show()

    print("natural frequencies:", frequencies)

if WIDGETS_AVAILABLE:
    interact(
        coupled_springs,
        m1=FloatSlider(min=0.5, max=4, step=0.25, value=1),
        m2=FloatSlider(min=0.5, max=4, step=0.25, value=1),
        k1=FloatSlider(min=0.5, max=10, step=0.5, value=4),
        k2=FloatSlider(min=0.5, max=10, step=0.5, value=2),
        x10=FloatSlider(min=-2, max=2, step=0.25, value=1),
        x20=FloatSlider(min=-2, max=2, step=0.25, value=0),
        final_time=IntSlider(min=10, max=60, step=5, value=30)
    )
else:
    coupled_springs()

## 2. Normal modes

A normal mode has the form

$$
\mathbf x(t)=\mathbf v\cos(\omega t),
$$

where

$$
(K-\omega^2M)\mathbf v=0.
$$

The eigenvectors determine relative mass motion.

In [ ]:
m1 = m2 = 1.0
k1, k2 = 4.0, 2.0
M = np.diag([m1, m2])
K = np.array([[k1+k2, -k2], [-k2, k2]])
vals, vecs = eig(K, M)
order = np.argsort(vals.real)
vals, vecs = vals[order].real, vecs[:, order].real

print("omega^2:", vals)
print("mode shapes:")
print(vecs)

t = np.linspace(0, 20, 1000)
for j in range(2):
    mode = vecs[:, j:j+1] * np.cos(np.sqrt(vals[j])*t)
    plt.plot(t, mode[0], label=r"$x_1$")
    plt.plot(t, mode[1], label=r"$x_2$")
    plt.title(f"Normal mode {j+1}")
    plt.legend()
    plt.show()

## 3. Beating from nearby modes

When two natural frequencies are close, a superposition can exhibit slow amplitude modulation.

In [ ]:
t = np.linspace(0, 80, 2500)
w1, w2 = 2.0, 2.15
signal = np.cos(w1*t)+np.cos(w2*t)
envelope = 2*np.cos((w2-w1)*t/2)

plt.plot(t, signal, label="superposition")
plt.plot(t, envelope, linestyle="--", label="envelope")
plt.plot(t, -envelope, linestyle="--")
plt.xlim(0, 80)
plt.legend()
plt.title("Beating from nearby frequencies")
plt.show()

## 4. First-order electrical network

A linear network can be written as

$$
\mathbf i'=A\mathbf i+\mathbf bE(t).
$$

Transforms reduce it to simultaneous algebraic equations.

In [ ]:
A = np.array([[-4.0, 1.5], [2.0, -3.0]])
bvec = np.array([3.0, 0.0])

def network_rhs(t, i):
    E = 10.0
    return A @ i + bvec*E

t = np.linspace(0, 8, 800)
sol = solve_ivp(network_rhs, (0, 8), [0, 0], t_eval=t, rtol=1e-10, atol=1e-12)
steady = -np.linalg.solve(A, bvec*10)

plt.plot(t, sol.y[0], label=r"$i_1$")
plt.plot(t, sol.y[1], label=r"$i_2$")
plt.axhline(steady[0], linestyle="--")
plt.axhline(steady[1], linestyle="--")
plt.legend()
plt.title("Two-current network response")
plt.show()

print("steady currents:", steady)

## Optional extension — Linearized double pendulum

A small-angle double pendulum has a matrix equation

$$
M\boldsymbol\theta''+K\boldsymbol\theta=0.
$$

Its two eigenmodes correspond approximately to in-phase and out-of-phase motion.

In [ ]:
m1, m2, l1, l2, g = 3.0, 1.0, 1.0, 1.0, 9.81
M = np.array([[(m1+m2)*l1**2, m2*l1*l2],
              [m2*l1*l2, m2*l2**2]])
K = np.array([[(m1+m2)*g*l1, 0],
              [0, m2*g*l2]])

vals, vecs = eig(K, M)
order = np.argsort(vals.real)
vals, vecs = vals[order].real, vecs[:, order].real
print("linearized frequencies:", np.sqrt(vals))
print("mode shapes:")
print(vecs)

In [ ]:
def double_pendulum_linear(theta10=0.3, theta20=-0.2, final_time=20):
    A = np.block([
        [np.zeros((2,2)), np.eye(2)],
        [-np.linalg.solve(M, K), np.zeros((2,2))]
    ])
    def rhs(t, z):
        return A @ z
    t = np.linspace(0, final_time, 1200)
    sol = solve_ivp(rhs, (0, final_time), [theta10, theta20, 0, 0],
                    t_eval=t, rtol=1e-9, atol=1e-11)
    plt.plot(t, sol.y[0], label=r"$\theta_1$")
    plt.plot(t, sol.y[1], label=r"$\theta_2$")
    plt.legend()
    plt.title("Linearized double-pendulum angles")
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        double_pendulum_linear,
        theta10=FloatSlider(min=-0.6, max=0.6, step=0.05, value=0.3),
        theta20=FloatSlider(min=-0.6, max=0.6, step=0.05, value=-0.2),
        final_time=IntSlider(min=5, max=40, step=5, value=20)
    )
else:
    double_pendulum_linear()

## Classroom Checkpoint — Exit Check

For

$$
\mathbf x'=A\mathbf x,
\qquad
\mathbf x(0)=\mathbf x_0,
$$

write the transformed equation.

> Pause here. Let students commit to an answer before running the next cell.